# 🎨 ComfyUI 배치 이미지 생성기 — Parksy

**사용법:**
1. 런타임 → GPU (T4) 확인
2. 전체 실행 (Ctrl+F9 또는 런타임→모두 실행)
3. Drive 마운트 허용
4. `MyDrive/ComfyUI_Batch/prompts.txt` 에 프롬프트 한 줄씩 작성
5. 결과: `MyDrive/ComfyUI_Batch/output/` 폴더

In [ ]:
# ─── 1. Google Drive 마운트 ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BATCH = '/content/drive/MyDrive/ComfyUI_Batch'
os.makedirs(f'{DRIVE_BATCH}/output', exist_ok=True)
print(f'✅ Drive 마운트 완료: {DRIVE_BATCH}')

In [ ]:
# ─── 2. ComfyUI 설치 ────────────────────────────────────────────────────
import subprocess, sys

if not os.path.exists('/content/ComfyUI'):
    print('📦 ComfyUI 클론 중...')
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI', '/content/ComfyUI'], check=True)
    print('📦 의존성 설치 중...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '/content/ComfyUI/requirements.txt', '-q'], check=True)
    print('✅ ComfyUI 설치 완료')
else:
    print('✅ ComfyUI 이미 설치됨')

In [ ]:
# ─── 3. SD 1.5 모델 다운로드 (~2GB, 5분) ──────────────────────────────────
MODEL_DIR = '/content/ComfyUI/models/checkpoints'
MODEL_FILE = f'{MODEL_DIR}/v1-5-pruned-emaonly.safetensors'

os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(MODEL_FILE):
    print('⬇️ SD v1.5 모델 다운로드 중... (약 2GB, 5분 소요)')
    subprocess.run([
        'wget', '-q', '--show-progress',
        '-O', MODEL_FILE,
        'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors'
    ], check=True)
    print('✅ 모델 다운로드 완료')
else:
    size = os.path.getsize(MODEL_FILE) / (1024**3)
    print(f'✅ 모델 이미 있음 ({size:.1f}GB)')

In [ ]:
# ─── 4. ComfyUI 서버 시작 ───────────────────────────────────────────────
import subprocess, time, threading

print('🚀 ComfyUI 서버 시작...')
server_proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    cwd='/content/ComfyUI'
)

# 서버 준비 대기
import requests as req
for _ in range(30):
    try:
        r = req.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if r.status_code == 200:
            print('✅ ComfyUI 서버 준비 완료')
            break
    except:
        pass
    time.sleep(2)
else:
    print('⚠️ 서버 시작 대기 중... 30초 후 재시도')

In [ ]:
# ─── 5. 배치 생성 함수 ────────────────────────────────────────────────────
import json, requests, time, shutil, random

def generate_image(prompt, negative_prompt='', steps=20, width=512, height=512, seed=None):
    """ComfyUI API로 이미지 1장 생성"""
    if seed is None:
        seed = random.randint(0, 2**32)
    
    workflow = {
        "3": {
            "class_type": "KSampler",
            "inputs": {
                "seed": seed, "steps": steps, "cfg": 7.0,
                "sampler_name": "euler", "scheduler": "normal", "denoise": 1.0,
                "model": ["4", 0], "positive": ["6", 0],
                "negative": ["7", 0], "latent_image": ["5", 0]
            }
        },
        "4": {
            "class_type": "CheckpointLoaderSimple",
            "inputs": {"ckpt_name": "v1-5-pruned-emaonly.safetensors"}
        },
        "5": {
            "class_type": "EmptyLatentImage",
            "inputs": {"width": width, "height": height, "batch_size": 1}
        },
        "6": {
            "class_type": "CLIPTextEncode",
            "inputs": {"text": prompt, "clip": ["4", 1]}
        },
        "7": {
            "class_type": "CLIPTextEncode",
            "inputs": {
                "text": negative_prompt or "ugly, deformed, bad anatomy, blurry, watermark",
                "clip": ["4", 1]
            }
        },
        "8": {
            "class_type": "VAEDecode",
            "inputs": {"samples": ["3", 0], "vae": ["4", 2]}
        },
        "9": {
            "class_type": "SaveImage",
            "inputs": {"filename_prefix": "batch", "images": ["8", 0]}
        }
    }
    
    r = requests.post('http://127.0.0.1:8188/prompt', json={'prompt': workflow})
    prompt_id = r.json()['prompt_id']
    
    # 완료 대기
    while True:
        hist = requests.get(f'http://127.0.0.1:8188/history/{prompt_id}').json()
        if prompt_id in hist:
            for node_id, out in hist[prompt_id]['outputs'].items():
                if 'images' in out:
                    return out['images'][0]
        time.sleep(1)

print('✅ 생성 함수 로드 완료')

In [ ]:
# ─── 6. 프롬프트 파일 확인 / 샘플 생성 ────────────────────────────────────
PROMPTS_FILE = f'{DRIVE_BATCH}/prompts.txt'
CONFIG_FILE  = f'{DRIVE_BATCH}/config.json'

# 기본 설정 파일
DEFAULT_CONFIG = {
    'steps': 20,
    'width': 512,
    'height': 512,
    'negative_prompt': 'ugly, deformed, bad anatomy, blurry, watermark'
}

if not os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, 'w') as f:
        json.dump(DEFAULT_CONFIG, f, indent=2, ensure_ascii=False)
    print(f'📝 config.json 생성: {CONFIG_FILE}')

if not os.path.exists(PROMPTS_FILE):
    sample_prompts = [
        'a beautiful sunset over mountains, golden hour, photorealistic, 8k',
        'a cozy cafe in autumn rain, watercolor style, warm colors',
        'futuristic city at night, neon lights, cyberpunk, cinematic',
        'korean street food market, bustling, vibrant colors, documentary photography',
        'ancient temple in misty forest, atmospheric, fantasy art'
    ]
    with open(PROMPTS_FILE, 'w', encoding='utf-8') as f:
        f.write('\n'.join(sample_prompts))
    print(f'📝 샘플 prompts.txt 생성: {PROMPTS_FILE}')
    print(f'   → 원하는 프롬프트로 수정 후 다음 셀 실행')
else:
    with open(PROMPTS_FILE, encoding='utf-8') as f:
        prompts = [l.strip() for l in f if l.strip()]
    print(f'✅ 프롬프트 {len(prompts)}개 로드됨')
    for i, p in enumerate(prompts):
        print(f'  {i+1}. {p[:60]}...' if len(p) > 60 else f'  {i+1}. {p}')

In [ ]:
# ─── 7. 배치 실행 ─────────────────────────────────────────────────────────
from IPython.display import display, Image as IPImage

with open(PROMPTS_FILE, encoding='utf-8') as f:
    prompts = [l.strip() for l in f if l.strip()]

with open(CONFIG_FILE) as f:
    cfg = json.load(f)

print(f'🎨 배치 시작: {len(prompts)}장')
print(f'   설정: {cfg["steps"]}step, {cfg["width"]}×{cfg["height"]}')
print(f'   예상 시간: 약 {len(prompts) * 15}초 (T4 GPU 기준)\n')

results = []
for i, prompt in enumerate(prompts):
    print(f'[{i+1}/{len(prompts)}] {prompt[:60]}')
    t0 = time.time()
    
    img_info = generate_image(
        prompt=prompt,
        negative_prompt=cfg.get('negative_prompt', ''),
        steps=cfg.get('steps', 20),
        width=cfg.get('width', 512),
        height=cfg.get('height', 512)
    )
    
    # ComfyUI 출력 → Drive 저장
    src = f"/content/ComfyUI/output/{img_info['filename']}"
    safe_prompt = prompt[:40].replace('/', '_').replace(' ', '_')
    dst = f"{DRIVE_BATCH}/output/{i+1:03d}_{safe_prompt}.png"
    shutil.copy(src, dst)
    
    elapsed = time.time() - t0
    print(f'  ✅ {elapsed:.1f}초 → {os.path.basename(dst)}')
    
    # 미리보기
    display(IPImage(src, width=256))
    results.append(dst)

print(f'\n🎉 완료! {len(results)}장 → {DRIVE_BATCH}/output/')

In [ ]:
# ─── (선택) 고급: 커스텀 워크플로우 JSON 사용 ─────────────────────────────
# ComfyUI 웹UI에서 내보낸 workflow_api.json을 
# Drive의 MyDrive/ComfyUI_Batch/workflow_api.json 에 올려두면
# 아래 코드로 그 워크플로우 그대로 실행 가능

CUSTOM_WORKFLOW = f'{DRIVE_BATCH}/workflow_api.json'

if os.path.exists(CUSTOM_WORKFLOW):
    with open(CUSTOM_WORKFLOW) as f:
        workflow = json.load(f)
    
    r = requests.post('http://127.0.0.1:8188/prompt', json={'prompt': workflow})
    prompt_id = r.json()['prompt_id']
    print(f'커스텀 워크플로우 실행: {prompt_id}')
    
    while True:
        hist = requests.get(f'http://127.0.0.1:8188/history/{prompt_id}').json()
        if prompt_id in hist:
            print('✅ 완료')
            break
        time.sleep(1)
else:
    print('ℹ️ workflow_api.json 없음 — 위 배치 셀 사용')